In [1]:
import csv
import json
import os
import random
import time
from copy import deepcopy
from dataclasses import asdict, dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


@dataclass(frozen=True)
class ExperimentConfig:
    num_basis: int = 32
    observation_dim: int = 128
    classes_per_task: int = 5
    samples_per_class_train: int = 500
    samples_per_class_test: int = 100
    latent_noise_std: float = 0.30
    observation_noise_std: float = 0.05

    shared_feature_dim: int = 32
    specialist_hidden_dim: int = 128
    specialist_dim: int = 64
    global_dim: int = 64
    expansion_dim: int = 64

    batch_size: int = 128
    novelty_batch_size: int = 256
    replay_per_class: int = 20

    base_epochs: int = 25
    probe_epochs: int = 20
    specialist_epochs: int = 30
    fusion_epochs: int = 40
    bias_epochs: int = 5

    base_lr: float = 1e-3
    probe_lr: float = 1e-3
    specialist_lr: float = 1e-3
    fusion_lr_new: float = 3e-3
    fusion_lr_old: float = 5e-4
    bias_lr: float = 5e-5
    weight_decay: float = 1e-4

    ce_weight: float = 2.5
    replay_weight: float = 0.8
    distillation_weight: float = 4.0
    feature_weight: float = 1.0
    distillation_temperature: float = 2.0

    threshold_values: tuple = (0.30, 0.50, 0.70)
    seeds: tuple = (42, 123, 456, 789, 1024)

    output_directory: str = "./results_controlled_capacity_reuse_corrected"


CONFIG = ExperimentConfig()


TASK_PROTOCOL = {
    0: {
        "basis_indices": tuple(range(0, 8)),
        "reference_task": None,
        "role": "base",
        "expected": "base",
    },
    1: {
        "basis_indices": tuple(range(0, 8)),
        "reference_task": 0,
        "role": "100% overlap with T0",
        "expected": "reuse",
    },
    2: {
        "basis_indices": tuple(range(4, 12)),
        "reference_task": 0,
        "role": "50% overlap with T0",
        "expected": "intermediate",
    },
    3: {
        "basis_indices": tuple(range(16, 24)),
        "reference_task": 0,
        "role": "0% overlap with T0",
        "expected": "expand",
    },
    4: {
        "basis_indices": tuple(range(16, 24)),
        "reference_task": 3,
        "role": "100% overlap with T3",
        "expected": "reuse",
    },
}


def seed_everything(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def reset_peak_memory(device):
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)


def get_peak_memory_mb(device):
    if device.type != "cuda":
        return 0.0

    return torch.cuda.max_memory_allocated(device) / (1024 ** 2)


class ControlledLatentDataGenerator:
    def __init__(self, config, seed):
        self.config = config
        self.rng = np.random.RandomState(seed)

        raw_matrix = self.rng.randn(
            config.observation_dim,
            config.num_basis,
        ).astype(np.float32)

        q, _ = np.linalg.qr(raw_matrix)

        self.observation_matrix = q[
            :,
            :config.num_basis,
        ].astype(np.float32)

        self.class_centers = {}

    def incoming_overlap(self, old_basis_indices, new_basis_indices):
        old_set = set(old_basis_indices)
        new_set = set(new_basis_indices)

        if len(new_set) == 0:
            return 0.0

        return len(old_set & new_set) / len(new_set)

    def generate_task(
        self,
        task_id,
        basis_indices,
        global_class_offset,
    ):
        config = self.config
        basis_indices = list(basis_indices)
        num_active_basis = len(basis_indices)

        centers = self.rng.randn(
            config.classes_per_task,
            num_active_basis,
        ).astype(np.float32)

        centers /= (
            np.linalg.norm(
                centers,
                axis=1,
                keepdims=True,
            )
            + 1e-8
        )

        centers *= 2.0

        self.class_centers[task_id] = {
            "basis_indices": basis_indices,
            "centers": centers,
        }

        def make_samples(samples_per_class):
            observations_all = []
            labels_all = []
            latents_all = []

            for class_id in range(config.classes_per_task):
                local_coefficients = (
                    centers[class_id][None, :]
                    + self.rng.randn(
                        samples_per_class,
                        num_active_basis,
                    ).astype(np.float32)
                    * config.latent_noise_std
                )

                latent_coefficients = np.zeros(
                    (
                        samples_per_class,
                        config.num_basis,
                    ),
                    dtype=np.float32,
                )

                latent_coefficients[
                    :,
                    basis_indices,
                ] = local_coefficients

                observations = (
                    latent_coefficients
                    @ self.observation_matrix.T
                )

                observations += (
                    self.rng.randn(
                        samples_per_class,
                        config.observation_dim,
                    ).astype(np.float32)
                    * config.observation_noise_std
                )

                labels = np.full(
                    samples_per_class,
                    class_id + global_class_offset,
                    dtype=np.int64,
                )

                observations_all.append(observations)
                labels_all.append(labels)
                latents_all.append(latent_coefficients)

            return (
                np.concatenate(observations_all, axis=0),
                np.concatenate(labels_all, axis=0),
                np.concatenate(latents_all, axis=0),
            )

        train_x, train_y, train_latent = make_samples(
            config.samples_per_class_train
        )

        test_x, test_y, test_latent = make_samples(
            config.samples_per_class_test
        )

        train_dataset = TensorDataset(
            torch.from_numpy(train_x),
            torch.from_numpy(train_y),
            torch.from_numpy(train_latent),
        )

        test_dataset = TensorDataset(
            torch.from_numpy(test_x),
            torch.from_numpy(test_y),
            torch.from_numpy(test_latent),
        )

        return train_dataset, test_dataset


class FixedSharedEncoder(nn.Module):
    def __init__(self, observation_matrix):
        super().__init__()

        inverse_mapping = torch.from_numpy(
            observation_matrix
        ).float()

        self.register_buffer(
            "inverse_mapping",
            inverse_mapping,
        )

    def forward(self, x):
        return x @ self.inverse_mapping


class SpecialistHead(nn.Module):
    def __init__(
        self,
        shared_feature_dim,
        hidden_dim,
        specialist_dim,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                shared_feature_dim,
                hidden_dim,
                bias=False,
            ),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(
                hidden_dim,
                specialist_dim,
                bias=False,
            ),
            nn.BatchNorm1d(specialist_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, shared_feature):
        return self.network(shared_feature)


class ChannelGating(nn.Module):
    def __init__(self, feature_dim):
        super().__init__()

        hidden_dim = max(feature_dim // 4, 4)

        self.network = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, feature_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.network(x)


class GlobalRepresentation(nn.Module):
    def __init__(
        self,
        specialist_dim,
        global_dim,
    ):
        super().__init__()

        self.specialist_dim = specialist_dim
        self.global_dim = global_dim

        self.projections = nn.ModuleDict()
        self.gates = nn.ModuleDict()

        self.bottleneck = nn.Sequential(
            nn.Linear(
                specialist_dim,
                global_dim,
                bias=False,
            ),
            nn.ReLU(inplace=True),
        )

    def add_specialist(self, task_id):
        key = str(task_id)

        self.projections[key] = nn.Linear(
            self.specialist_dim,
            self.specialist_dim,
            bias=False,
        )

        self.gates[key] = ChannelGating(
            self.specialist_dim
        )

    def forward(self, specialist_outputs):
        if len(specialist_outputs) == 0:
            return None

        projected_features = []

        for task_id, feature in specialist_outputs.items():
            key = str(task_id)

            if key not in self.projections:
                continue

            projected = self.projections[key](feature)
            gated = self.gates[key](projected)

            projected_features.append(gated)

        if len(projected_features) == 0:
            return None

        merged = torch.stack(
            projected_features,
            dim=0,
        ).mean(dim=0)

        return self.bottleneck(merged)


class FusionModule(nn.Module):
    def __init__(
        self,
        specialist_dim,
        global_dim,
        expansion_dim,
    ):
        super().__init__()

        self.expansion_dim = expansion_dim

        self.reuse_mapping = nn.Sequential(
            nn.Linear(
                specialist_dim,
                global_dim,
                bias=False,
            ),
            nn.BatchNorm1d(global_dim),
            nn.ReLU(inplace=True),
        )

        self.expand_mapping = None

        if expansion_dim > 0:
            self.expand_mapping = nn.Sequential(
                nn.Linear(
                    specialist_dim,
                    expansion_dim,
                    bias=False,
                ),
                nn.BatchNorm1d(expansion_dim),
                nn.ReLU(inplace=True),
            )

        self.gate = nn.Parameter(torch.tensor(0.0))

    def forward(
        self,
        specialist_feature,
        old_global_feature,
    ):
        reusable_delta = self.reuse_mapping(
            specialist_feature
        )

        gate_value = torch.sigmoid(self.gate)

        updated_old = (
            old_global_feature
            + gate_value * reusable_delta
        )

        if self.expand_mapping is None:
            return updated_old

        expanded = self.expand_mapping(
            specialist_feature
        )

        return torch.cat(
            [
                updated_old,
                expanded,
            ],
            dim=1,
        )


class CosineClassifier(nn.Module):
    def __init__(
        self,
        in_features,
        num_classes,
        temperature_init=10.0,
    ):
        super().__init__()

        self.weight = nn.Parameter(
            torch.empty(
                num_classes,
                in_features,
            )
        )

        nn.init.kaiming_normal_(self.weight)

        self.temperature = nn.Parameter(
            torch.tensor(float(temperature_init))
        )

    def forward(self, x):
        return self.temperature * F.linear(
            F.normalize(x, dim=1),
            F.normalize(self.weight, dim=1),
        )

    def expand_classes(self, num_new_classes):
        if num_new_classes < 0:
            raise ValueError(
                "num_new_classes must be non-negative"
            )

        if num_new_classes == 0:
            return

        old_weight = self.weight

        old_num_classes, feature_dim = old_weight.shape

        new_weight = nn.Parameter(
            torch.empty(
                old_num_classes + num_new_classes,
                feature_dim,
                device=old_weight.device,
                dtype=old_weight.dtype,
            )
        )

        nn.init.kaiming_normal_(new_weight)

        with torch.no_grad():
            new_weight[:old_num_classes].copy_(
                old_weight
            )

        self.weight = new_weight

    def expand_features(self, new_feature_dim):
        old_weight = self.weight

        num_classes, old_feature_dim = old_weight.shape

        if new_feature_dim < old_feature_dim:
            raise ValueError(
                "new_feature_dim must not shrink"
            )

        if new_feature_dim == old_feature_dim:
            return

        new_weight = nn.Parameter(
            torch.empty(
                num_classes,
                new_feature_dim,
                device=old_weight.device,
                dtype=old_weight.dtype,
            )
        )

        nn.init.kaiming_normal_(new_weight)

        with torch.no_grad():
            new_weight[
                :,
                :old_feature_dim,
            ].copy_(old_weight)

        self.weight = new_weight


class KFNControlled(nn.Module):
    def __init__(
        self,
        observation_matrix,
        num_initial_classes,
        config,
        device,
    ):
        super().__init__()

        self.config = config
        self.device = torch.device(device)

        self.shared_encoder = FixedSharedEncoder(
            observation_matrix
        )

        self.specialists = nn.ModuleDict()
        self.fusions = nn.ModuleDict()

        self.global_representation = GlobalRepresentation(
            config.specialist_dim,
            config.global_dim,
        )

        self.feature_dim = config.global_dim

        self.classifier = CosineClassifier(
            config.global_dim,
            num_initial_classes,
        )

        self.num_tasks = 0
        self.expansion_history = {}

    def encode_shared(self, x):
        return self.shared_encoder(x)

    def add_task(
        self,
        task_id,
        expansion_dim,
        num_new_classes,
    ):
        key = str(task_id)

        self.specialists[key] = SpecialistHead(
            self.config.shared_feature_dim,
            self.config.specialist_hidden_dim,
            self.config.specialist_dim,
        ).to(self.device)

        if task_id > 0:
            self.global_representation.add_specialist(
                task_id - 1
            )

            self.global_representation.projections[
                str(task_id - 1)
            ].to(self.device)

            self.global_representation.gates[
                str(task_id - 1)
            ].to(self.device)

        self.fusions[key] = FusionModule(
            self.config.specialist_dim,
            self.config.global_dim,
            expansion_dim,
        ).to(self.device)

        if expansion_dim > 0:
            self.feature_dim += expansion_dim

            self.classifier.expand_features(
                self.feature_dim
            )

        if num_new_classes > 0:
            self.classifier.expand_classes(
                num_new_classes
            )

        self.classifier.to(self.device)

        self.expansion_history[task_id] = expansion_dim
        self.num_tasks += 1

    def specialist_features(
        self,
        x,
        task_id,
    ):
        shared_features = self.encode_shared(x)

        return self.specialists[str(task_id)](
            shared_features
        )

    def forward(
        self,
        x,
        up_to=None,
        return_old_feature=False,
    ):
        if up_to is None:
            up_to = self.num_tasks - 1

        shared_features = self.encode_shared(x)

        specialist_outputs = {
            task_id: self.specialists[str(task_id)](
                shared_features
            )
            for task_id in range(up_to + 1)
        }

        if up_to == 0:
            old_feature = None

            fused_feature = self.fusions["0"].reuse_mapping(
                specialist_outputs[0]
            )
        else:
            old_outputs = {
                task_id: feature
                for task_id, feature in specialist_outputs.items()
                if task_id < up_to
            }

            old_feature = self.global_representation(
                old_outputs
            )

            if old_feature is None:
                old_feature = torch.zeros(
                    x.size(0),
                    self.config.global_dim,
                    device=x.device,
                    dtype=x.dtype,
                )

            fused_feature = self.fusions[str(up_to)](
                specialist_outputs[up_to],
                old_feature,
            )

        if fused_feature.size(1) < self.feature_dim:
            fused_feature = F.pad(
                fused_feature,
                (
                    0,
                    self.feature_dim
                    - fused_feature.size(1),
                ),
            )
        else:
            fused_feature = fused_feature[
                :,
                :self.feature_dim,
            ]

        logits = self.classifier(fused_feature)

        if return_old_feature:
            return logits, old_feature

        return logits

    def compute_shared_space_novelty(
        self,
        dataloader,
        old_basis_indices,
    ):
        if len(old_basis_indices) == 0:
            return 1.0

        old_basis_indices = list(old_basis_indices)

        self.eval()

        scores = []

        with torch.no_grad():
            for batch in dataloader:
                x = batch[0].to(self.device)

                shared_features = self.encode_shared(x)

                old_component = torch.zeros_like(
                    shared_features
                )

                old_component[
                    :,
                    old_basis_indices,
                ] = shared_features[
                    :,
                    old_basis_indices,
                ]

                residual = (
                    shared_features
                    - old_component
                )

                score = residual.norm(dim=1) / (
                    shared_features.norm(dim=1)
                    + 1e-8
                )

                scores.append(score)

        return float(
            torch.cat(scores, dim=0).mean().item()
        )

    def compute_exact_latent_novelty(
        self,
        dataloader,
        old_basis_indices,
    ):
        if len(old_basis_indices) == 0:
            return 1.0

        old_basis_indices = list(old_basis_indices)

        self.eval()

        scores = []

        with torch.no_grad():
            for batch in dataloader:
                latent_features = batch[2].to(self.device)

                old_component = torch.zeros_like(
                    latent_features
                )

                old_component[
                    :,
                    old_basis_indices,
                ] = latent_features[
                    :,
                    old_basis_indices,
                ]

                residual = (
                    latent_features
                    - old_component
                )

                score = residual.norm(dim=1) / (
                    latent_features.norm(dim=1)
                    + 1e-8
                )

                scores.append(score)

        return float(
            torch.cat(scores, dim=0).mean().item()
        )


class ReplayBuffer:
    def __init__(self, max_per_class):
        self.max_per_class = max_per_class
        self.buffer = {}

    def update(
        self,
        dataset,
        class_labels,
    ):
        for class_label in class_labels:
            examples = [
                (
                    dataset[index][0],
                    dataset[index][1],
                )
                for index in range(len(dataset))
                if dataset[index][1].item()
                == class_label
            ]

            if len(examples) > self.max_per_class:
                examples = random.sample(
                    examples,
                    self.max_per_class,
                )

            self.buffer[class_label] = examples

    def sample(self, num_samples):
        all_examples = [
            example
            for examples in self.buffer.values()
            for example in examples
        ]

        if len(all_examples) == 0:
            return None, None

        selected = random.choices(
            all_examples,
            k=min(num_samples, len(all_examples)),
        )

        x = torch.stack(
            [example[0] for example in selected]
        )

        y = torch.tensor(
            [example[1] for example in selected],
            dtype=torch.long,
        )

        return x, y


def collect_old_basis_indices(task_id):
    old_basis_indices = set()

    for previous_task_id in range(task_id):
        old_basis_indices.update(
            TASK_PROTOCOL[previous_task_id][
                "basis_indices"
            ]
        )

    return tuple(sorted(old_basis_indices))


def choose_expansion_dim(
    policy,
    novelty,
    threshold,
    config,
):
    if policy == "always_expand":
        return config.expansion_dim

    if policy == "always_reuse":
        return 0

    if policy == "adaptive":
        return (
            config.expansion_dim
            if novelty > threshold
            else 0
        )

    raise ValueError(
        f"Unknown policy: {policy}"
    )


def parameter_breakdown(model):
    shared_encoder_params = sum(
        parameter.numel()
        for parameter in model.shared_encoder.parameters()
    )

    specialist_params = sum(
        parameter.numel()
        for specialist in model.specialists.values()
        for parameter in specialist.parameters()
    )

    fusion_params = sum(
        parameter.numel()
        for fusion in model.fusions.values()
        for parameter in fusion.parameters()
    )

    global_params = sum(
        parameter.numel()
        for parameter in model.global_representation.parameters()
    )

    classifier_params = sum(
        parameter.numel()
        for parameter in model.classifier.parameters()
    )

    total_params = (
        shared_encoder_params
        + specialist_params
        + fusion_params
        + global_params
        + classifier_params
    )

    return {
        "shared_encoder": int(shared_encoder_params),
        "specialists": int(specialist_params),
        "fusion": int(fusion_params),
        "global_representation": int(global_params),
        "classifier": int(classifier_params),
        "total": int(total_params),
    }


def train_base(
    model,
    loader,
    config,
    device,
):
    optimizer = optim.AdamW(
        model.parameters(),
        lr=config.base_lr,
        weight_decay=config.weight_decay,
    )

    model.train()

    for _ in range(config.base_epochs):
        for batch in loader:
            x, y = batch[:2]

            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits = model(x, up_to=0)

            loss = F.cross_entropy(
                logits,
                y,
            )

            loss.backward()
            optimizer.step()


def train_specialist_probe(
    model,
    task_id,
    loader,
    task_offset,
    config,
    device,
):
    specialist = model.specialists[str(task_id)]

    probe_head = CosineClassifier(
        config.specialist_dim,
        config.classes_per_task,
    ).to(device)

    optimizer = optim.AdamW(
        list(specialist.parameters())
        + list(probe_head.parameters()),
        lr=config.probe_lr,
        weight_decay=config.weight_decay,
    )

    specialist.train()
    probe_head.train()

    for _ in range(config.probe_epochs):
        for batch in loader:
            x, y = batch[:2]

            x = x.to(device)
            y = (y - task_offset).to(device)

            optimizer.zero_grad()

            shared_features = model.encode_shared(x)

            specialist_features = specialist(
                shared_features
            )

            logits = probe_head(specialist_features)

            loss = F.cross_entropy(
                logits,
                y,
            )

            loss.backward()
            optimizer.step()


def train_specialist_phase(
    model,
    task_id,
    loader,
    config,
    device,
):
    specialist = model.specialists[str(task_id)]

    optimizer = optim.AdamW(
        specialist.parameters(),
        lr=config.specialist_lr,
        weight_decay=config.weight_decay,
    )

    specialist.train()

    for _ in range(config.specialist_epochs):
        for batch in loader:
            x, y = batch[:2]

            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            shared_features = model.encode_shared(x)

            specialist_features = specialist(
                shared_features
            )

            if specialist_features.size(1) < model.feature_dim:
                classifier_features = F.pad(
                    specialist_features,
                    (
                        0,
                        model.feature_dim
                        - specialist_features.size(1),
                    ),
                )
            else:
                classifier_features = specialist_features[
                    :,
                    :model.feature_dim,
                ]

            logits = model.classifier(
                classifier_features
            )

            loss = F.cross_entropy(
                logits,
                y,
            )

            loss.backward()
            optimizer.step()


def train_fusion_phase(
    model,
    task_id,
    loader,
    replay_buffer,
    old_model,
    config,
    device,
):
    for previous_task_id in range(task_id):
        for parameter in model.specialists[
            str(previous_task_id)
        ].parameters():
            parameter.requires_grad = False

    new_parameters = list(
        model.fusions[str(task_id)].parameters()
    )

    new_parameters.extend(
        model.classifier.parameters()
    )

    for key in model.global_representation.projections:
        new_parameters.extend(
            model.global_representation.projections[
                key
            ].parameters()
        )

    old_parameters = list(
        model.global_representation.bottleneck.parameters()
    )

    for key in model.global_representation.gates:
        for parameter in model.global_representation.gates[
            key
        ].parameters():
            parameter.requires_grad = False

    optimizer = optim.AdamW(
        [
            {
                "params": new_parameters,
                "lr": config.fusion_lr_new,
                "weight_decay": config.weight_decay,
            },
            {
                "params": old_parameters,
                "lr": config.fusion_lr_old,
                "weight_decay": config.weight_decay,
            },
        ]
    )

    model.train()

    for epoch in range(config.fusion_epochs):
        if epoch >= config.fusion_epochs // 2:
            for key in model.global_representation.gates:
                for parameter in model.global_representation.gates[
                    key
                ].parameters():
                    parameter.requires_grad = True

        for batch in loader:
            x, y = batch[:2]

            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits, current_old_feature = model(
                x,
                up_to=task_id,
                return_old_feature=True,
            )

            loss = config.ce_weight * F.cross_entropy(
                logits,
                y,
            )

            replay_x, replay_y = replay_buffer.sample(
                x.size(0)
            )

            if replay_x is not None:
                replay_x = replay_x.to(device)
                replay_y = replay_y.to(device)

                replay_logits = model(
                    replay_x,
                    up_to=task_id,
                )

                replay_loss = F.cross_entropy(
                    replay_logits,
                    replay_y,
                )

                loss = (
                    loss
                    + config.replay_weight * replay_loss
                )

            if old_model is not None:
                with torch.no_grad():
                    old_logits, frozen_old_feature = old_model(
                        x,
                        up_to=task_id - 1,
                        return_old_feature=True,
                    )

                current_old_logits = logits[
                    :,
                    :old_logits.size(1),
                ]

                temperature = (
                    config.distillation_temperature
                )

                distillation_loss = F.kl_div(
                    F.log_softmax(
                        current_old_logits / temperature,
                        dim=1,
                    ),
                    F.softmax(
                        old_logits / temperature,
                        dim=1,
                    ),
                    reduction="batchmean",
                )

                loss = (
                    loss
                    + config.distillation_weight
                    * distillation_loss
                    * temperature
                    * temperature
                )

                if (
                    current_old_feature is not None
                    and frozen_old_feature is not None
                ):
                    feature_loss = F.mse_loss(
                        current_old_feature,
                        frozen_old_feature,
                    )

                    loss = (
                        loss
                        + config.feature_weight
                        * feature_loss
                    )

            loss.backward()
            optimizer.step()

    for parameter in model.parameters():
        parameter.requires_grad = True


def train_bias_correction(
    model,
    task_id,
    replay_buffer,
    config,
    device,
):
    for parameter in model.parameters():
        parameter.requires_grad = False

    model.classifier.weight.requires_grad = True
    model.classifier.temperature.requires_grad = True

    optimizer = optim.AdamW(
        [
            model.classifier.weight,
            model.classifier.temperature,
        ],
        lr=config.bias_lr,
    )

    model.train()

    for _ in range(config.bias_epochs):
        replay_x, replay_y = replay_buffer.sample(128)

        if replay_x is None:
            break

        replay_x = replay_x.to(device)
        replay_y = replay_y.to(device)

        optimizer.zero_grad()

        logits = model(
            replay_x,
            up_to=task_id,
        )

        loss = F.cross_entropy(
            logits,
            replay_y,
        )

        loss.backward()
        optimizer.step()

    for parameter in model.parameters():
        parameter.requires_grad = True


def evaluate_all(
    model,
    test_loaders,
    device,
):
    model.eval()

    results = {}

    with torch.no_grad():
        for task_id, loader in test_loaders.items():
            correct = 0
            total = 0

            for batch in loader:
                x, y = batch[:2]

                x = x.to(device)
                y = y.to(device)

                logits = model(
                    x,
                    up_to=model.num_tasks - 1,
                )

                predictions = logits.argmax(dim=1)

                correct += (
                    predictions == y
                ).sum().item()

                total += y.size(0)

            results[task_id] = (
                100.0 * correct / total
                if total > 0
                else 0.0
            )

    model.train()

    return results


def compute_metrics(
    accuracy_matrix,
    num_tasks,
):
    final_accuracies = [
        accuracy_matrix[num_tasks - 1].get(
            task_id,
            0.0,
        )
        for task_id in range(num_tasks)
    ]

    diagonal_accuracies = [
        accuracy_matrix[task_id].get(
            task_id,
            0.0,
        )
        for task_id in range(num_tasks)
    ]

    forgetting_scores = []

    for task_id in range(num_tasks - 1):
        peak_accuracy = max(
            accuracy_matrix[stage].get(
                task_id,
                0.0,
            )
            for stage in range(task_id, num_tasks)
        )

        final_accuracy = accuracy_matrix[
            num_tasks - 1
        ].get(task_id, 0.0)

        forgetting_scores.append(
            peak_accuracy - final_accuracy
        )

    backward_transfer_scores = [
        accuracy_matrix[num_tasks - 1].get(
            task_id,
            0.0,
        )
        - accuracy_matrix[task_id].get(
            task_id,
            0.0,
        )
        for task_id in range(num_tasks - 1)
    ]

    return {
        "avg_accuracy": float(
            np.mean(final_accuracies)
        ),
        "min_accuracy": float(
            np.min(final_accuracies)
        ),
        "plasticity": float(
            np.mean(diagonal_accuracies)
        ),
        "forgetting": (
            float(np.mean(forgetting_scores))
            if forgetting_scores
            else 0.0
        ),
        "bwt": (
            float(np.mean(backward_transfer_scores))
            if backward_transfer_scores
            else 0.0
        ),
        "std_accuracy": float(
            np.std(final_accuracies)
        ),
    }


def expected_clear_decision(task_id):
    expected = TASK_PROTOCOL[task_id]["expected"]

    if expected in {"reuse", "expand"}:
        return expected

    return "intermediate"


def run_single(
    policy,
    threshold,
    seed,
    config,
    device,
):
    seed_everything(seed)

    device = torch.device(device)

    generator = ControlledLatentDataGenerator(
        config,
        seed,
    )

    train_sets = {}
    test_sets = {}
    offsets = {}

    class_offset = 0

    for task_id, task_info in TASK_PROTOCOL.items():
        offsets[task_id] = class_offset

        train_dataset, test_dataset = (
            generator.generate_task(
                task_id,
                task_info["basis_indices"],
                class_offset,
            )
        )

        train_sets[task_id] = train_dataset
        test_sets[task_id] = test_dataset

        class_offset += config.classes_per_task

    model = KFNControlled(
        observation_matrix=generator.observation_matrix,
        num_initial_classes=config.classes_per_task,
        config=config,
        device=device,
    ).to(device)

    replay_buffer = ReplayBuffer(
        config.replay_per_class
    )

    accuracy_matrix = {}
    task_records = []
    parameter_history = []
    timing_history = []
    memory_history = []

    for task_id, task_info in TASK_PROTOCOL.items():
        reset_peak_memory(device)
        synchronize(device)

        start_time = time.time()

        train_loader = DataLoader(
            train_sets[task_id],
            batch_size=config.batch_size,
            shuffle=True,
        )

        novelty_loader = DataLoader(
            train_sets[task_id],
            batch_size=config.novelty_batch_size,
            shuffle=False,
        )

        if task_id == 0:
            model.add_task(
                task_id=0,
                expansion_dim=0,
                num_new_classes=0,
            )

            train_base(
                model,
                train_loader,
                config,
                device,
            )

            incoming_overlap = None
            exact_latent_novelty = 1.0
            learned_novelty = 1.0
            gate_decision = "base"
            expansion_dim = 0

        else:
            model.specialists[str(task_id)] = SpecialistHead(
                config.shared_feature_dim,
                config.specialist_hidden_dim,
                config.specialist_dim,
            ).to(device)

            train_specialist_probe(
                model,
                task_id,
                train_loader,
                offsets[task_id],
                config,
                device,
            )

            old_basis_indices = collect_old_basis_indices(
                task_id
            )

            exact_latent_novelty = (
                model.compute_exact_latent_novelty(
                    novelty_loader,
                    old_basis_indices,
                )
            )

            learned_novelty = (
                model.compute_shared_space_novelty(
                    novelty_loader,
                    old_basis_indices,
                )
            )

            expansion_dim = choose_expansion_dim(
                policy,
                learned_novelty,
                threshold,
                config,
            )

            gate_decision = (
                "expand"
                if expansion_dim > 0
                else "reuse"
            )

            reference_task = task_info["reference_task"]

            incoming_overlap = (
                generator.incoming_overlap(
                    TASK_PROTOCOL[reference_task][
                        "basis_indices"
                    ],
                    task_info["basis_indices"],
                )
                if reference_task is not None
                else None
            )

            del model.specialists[str(task_id)]

            old_model = deepcopy(model).to(device)
            old_model.eval()

            for parameter in old_model.parameters():
                parameter.requires_grad = False

            model.add_task(
                task_id=task_id,
                expansion_dim=expansion_dim,
                num_new_classes=config.classes_per_task,
            )

            train_specialist_phase(
                model,
                task_id,
                train_loader,
                config,
                device,
            )

            train_fusion_phase(
                model,
                task_id,
                train_loader,
                replay_buffer,
                old_model,
                config,
                device,
            )

            train_bias_correction(
                model,
                task_id,
                replay_buffer,
                config,
                device,
            )

            del old_model

        replay_buffer.update(
            train_sets[task_id],
            list(
                range(
                    offsets[task_id],
                    offsets[task_id]
                    + config.classes_per_task,
                )
            ),
        )

        test_loaders = {
            previous_task_id: DataLoader(
                test_sets[previous_task_id],
                batch_size=config.novelty_batch_size,
                shuffle=False,
            )
            for previous_task_id in range(task_id + 1)
        }

        accuracies = evaluate_all(
            model,
            test_loaders,
            device,
        )

        accuracy_matrix[task_id] = accuracies

        synchronize(device)

        task_time_seconds = time.time() - start_time
        peak_memory_mb = get_peak_memory_mb(device)
        parameters = parameter_breakdown(model)

        parameter_history.append(parameters)
        timing_history.append(task_time_seconds)
        memory_history.append(peak_memory_mb)

        record = {
            "task_id": task_id,
            "role": task_info["role"],
            "reference_task": task_info["reference_task"],
            "basis_indices": list(
                task_info["basis_indices"]
            ),
            "incoming_overlap": incoming_overlap,
            "exact_latent_novelty": exact_latent_novelty,
            "learned_novelty": learned_novelty,
            "policy": policy,
            "threshold": threshold,
            "expected": expected_clear_decision(task_id),
            "gate_decision": gate_decision,
            "expansion_dim": expansion_dim,
            "parameters": parameters,
            "task_time_seconds": task_time_seconds,
            "peak_memory_mb": peak_memory_mb,
            "accuracies": accuracies,
        }

        task_records.append(record)

        overlap_text = (
            "N/A"
            if incoming_overlap is None
            else f"{incoming_overlap:.2f}"
        )

        accuracy_text = ", ".join(
            f"T{previous_task_id}:{accuracy:.1f}"
            for previous_task_id, accuracy
            in accuracies.items()
        )

        print(
            f"  T{task_id} "
            f"[{task_info['role']:<22}] "
            f"overlap={overlap_text:<4} "
            f"latent_nu={exact_latent_novelty:.4f} "
            f"learned_nu={learned_novelty:.4f} "
            f"gate={gate_decision:<6} "
            f"d_new={expansion_dim:<3} "
            f"params={parameters['total'] / 1e6:.3f}M "
            f"time={task_time_seconds:.1f}s "
            f"memory={peak_memory_mb:.1f}MB "
            f"accs={{{accuracy_text}}}"
        )

    metrics = compute_metrics(
        accuracy_matrix,
        len(TASK_PROTOCOL),
    )

    clear_task_ids = [1, 3, 4]

    correct_clear_decisions = []

    for task_id in clear_task_ids:
        expected = expected_clear_decision(task_id)

        observed = next(
            record["gate_decision"]
            for record in task_records
            if record["task_id"] == task_id
        )

        correct_clear_decisions.append(
            int(expected == observed)
        )

    metrics.update(
        {
            "policy": policy,
            "threshold": threshold,
            "seed": seed,
            "task_records": task_records,
            "accuracy_matrix": {
                str(task_id): values
                for task_id, values
                in accuracy_matrix.items()
            },
            "parameter_history": parameter_history,
            "timing_history_seconds": timing_history,
            "memory_history_mb": memory_history,
            "final_parameters": parameter_history[-1],
            "total_time_seconds": float(
                sum(timing_history)
            ),
            "peak_memory_mb": float(
                max(memory_history)
            ),
            "num_expansions": int(
                sum(
                    record["expansion_dim"] > 0
                    for record in task_records
                )
            ),
            "clear_gate_accuracy": float(
                np.mean(correct_clear_decisions)
            ),
        }
    )

    return metrics


def mean_std(values):
    values = np.asarray(values, dtype=np.float64)

    return {
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
    }


def aggregate_results(all_results):
    aggregated = {}

    for configuration_name, seed_results in all_results.items():
        aggregated[configuration_name] = {
            "avg_accuracy": mean_std(
                [
                    result["avg_accuracy"]
                    for result in seed_results
                ]
            ),
            "min_accuracy": mean_std(
                [
                    result["min_accuracy"]
                    for result in seed_results
                ]
            ),
            "forgetting": mean_std(
                [
                    result["forgetting"]
                    for result in seed_results
                ]
            ),
            "plasticity": mean_std(
                [
                    result["plasticity"]
                    for result in seed_results
                ]
            ),
            "bwt": mean_std(
                [
                    result["bwt"]
                    for result in seed_results
                ]
            ),
            "std_accuracy": mean_std(
                [
                    result["std_accuracy"]
                    for result in seed_results
                ]
            ),
            "num_expansions": mean_std(
                [
                    result["num_expansions"]
                    for result in seed_results
                ]
            ),
            "clear_gate_accuracy": mean_std(
                [
                    result["clear_gate_accuracy"]
                    for result in seed_results
                ]
            ),
            "total_time_seconds": mean_std(
                [
                    result["total_time_seconds"]
                    for result in seed_results
                ]
            ),
            "peak_memory_mb": mean_std(
                [
                    result["peak_memory_mb"]
                    for result in seed_results
                ]
            ),
            "final_total_params": mean_std(
                [
                    result["final_parameters"]["total"]
                    for result in seed_results
                ]
            ),
            "final_specialist_params": mean_std(
                [
                    result["final_parameters"][
                        "specialists"
                    ]
                    for result in seed_results
                ]
            ),
            "final_fusion_params": mean_std(
                [
                    result["final_parameters"]["fusion"]
                    for result in seed_results
                ]
            ),
            "final_global_params": mean_std(
                [
                    result["final_parameters"][
                        "global_representation"
                    ]
                    for result in seed_results
                ]
            ),
            "final_classifier_params": mean_std(
                [
                    result["final_parameters"][
                        "classifier"
                    ]
                    for result in seed_results
                ]
            ),
        }

    return aggregated


def write_summary_csv(
    aggregated_results,
    output_directory,
):
    output_path = os.path.join(
        output_directory,
        "summary.csv",
    )

    fieldnames = [
        "configuration",
        "avg_accuracy_mean",
        "avg_accuracy_std",
        "min_accuracy_mean",
        "min_accuracy_std",
        "forgetting_mean",
        "forgetting_std",
        "plasticity_mean",
        "plasticity_std",
        "bwt_mean",
        "bwt_std",
        "num_expansions_mean",
        "num_expansions_std",
        "clear_gate_accuracy_mean",
        "clear_gate_accuracy_std",
        "final_total_params_mean",
        "final_total_params_std",
        "final_specialist_params_mean",
        "final_fusion_params_mean",
        "final_global_params_mean",
        "final_classifier_params_mean",
        "peak_memory_mb_mean",
        "peak_memory_mb_std",
        "total_time_seconds_mean",
        "total_time_seconds_std",
    ]

    with open(
        output_path,
        "w",
        newline="",
        encoding="utf-8",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=fieldnames,
        )

        writer.writeheader()

        for configuration_name, values in (
            aggregated_results.items()
        ):
            writer.writerow(
                {
                    "configuration": configuration_name,
                    "avg_accuracy_mean": values[
                        "avg_accuracy"
                    ]["mean"],
                    "avg_accuracy_std": values[
                        "avg_accuracy"
                    ]["std"],
                    "min_accuracy_mean": values[
                        "min_accuracy"
                    ]["mean"],
                    "min_accuracy_std": values[
                        "min_accuracy"
                    ]["std"],
                    "forgetting_mean": values[
                        "forgetting"
                    ]["mean"],
                    "forgetting_std": values[
                        "forgetting"
                    ]["std"],
                    "plasticity_mean": values[
                        "plasticity"
                    ]["mean"],
                    "plasticity_std": values[
                        "plasticity"
                    ]["std"],
                    "bwt_mean": values["bwt"]["mean"],
                    "bwt_std": values["bwt"]["std"],
                    "num_expansions_mean": values[
                        "num_expansions"
                    ]["mean"],
                    "num_expansions_std": values[
                        "num_expansions"
                    ]["std"],
                    "clear_gate_accuracy_mean": values[
                        "clear_gate_accuracy"
                    ]["mean"],
                    "clear_gate_accuracy_std": values[
                        "clear_gate_accuracy"
                    ]["std"],
                    "final_total_params_mean": values[
                        "final_total_params"
                    ]["mean"],
                    "final_total_params_std": values[
                        "final_total_params"
                    ]["std"],
                    "final_specialist_params_mean": values[
                        "final_specialist_params"
                    ]["mean"],
                    "final_fusion_params_mean": values[
                        "final_fusion_params"
                    ]["mean"],
                    "final_global_params_mean": values[
                        "final_global_params"
                    ]["mean"],
                    "final_classifier_params_mean": values[
                        "final_classifier_params"
                    ]["mean"],
                    "peak_memory_mb_mean": values[
                        "peak_memory_mb"
                    ]["mean"],
                    "peak_memory_mb_std": values[
                        "peak_memory_mb"
                    ]["std"],
                    "total_time_seconds_mean": values[
                        "total_time_seconds"
                    ]["mean"],
                    "total_time_seconds_std": values[
                        "total_time_seconds"
                    ]["std"],
                }
            )


def write_transition_csv(
    all_results,
    output_directory,
):
    output_path = os.path.join(
        output_directory,
        "transitions.csv",
    )

    fieldnames = [
        "configuration",
        "seed",
        "task_id",
        "role",
        "reference_task",
        "basis_indices",
        "incoming_overlap",
        "exact_latent_novelty",
        "learned_novelty",
        "policy",
        "threshold",
        "expected",
        "gate_decision",
        "expansion_dim",
        "task_time_seconds",
        "peak_memory_mb",
        "shared_encoder_params",
        "specialist_params",
        "fusion_params",
        "global_representation_params",
        "classifier_params",
        "total_params",
    ]

    with open(
        output_path,
        "w",
        newline="",
        encoding="utf-8",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=fieldnames,
        )

        writer.writeheader()

        for configuration_name, seed_results in (
            all_results.items()
        ):
            for result in seed_results:
                for record in result["task_records"]:
                    params = record["parameters"]

                    writer.writerow(
                        {
                            "configuration": configuration_name,
                            "seed": result["seed"],
                            "task_id": record["task_id"],
                            "role": record["role"],
                            "reference_task": record[
                                "reference_task"
                            ],
                            "basis_indices": json.dumps(
                                record["basis_indices"]
                            ),
                            "incoming_overlap": record[
                                "incoming_overlap"
                            ],
                            "exact_latent_novelty": record[
                                "exact_latent_novelty"
                            ],
                            "learned_novelty": record[
                                "learned_novelty"
                            ],
                            "policy": record["policy"],
                            "threshold": record["threshold"],
                            "expected": record["expected"],
                            "gate_decision": record[
                                "gate_decision"
                            ],
                            "expansion_dim": record[
                                "expansion_dim"
                            ],
                            "task_time_seconds": record[
                                "task_time_seconds"
                            ],
                            "peak_memory_mb": record[
                                "peak_memory_mb"
                            ],
                            "shared_encoder_params": params[
                                "shared_encoder"
                            ],
                            "specialist_params": params[
                                "specialists"
                            ],
                            "fusion_params": params[
                                "fusion"
                            ],
                            "global_representation_params": params[
                                "global_representation"
                            ],
                            "classifier_params": params[
                                "classifier"
                            ],
                            "total_params": params["total"],
                        }
                    )


def print_protocol():
    print("=" * 100)
    print(
        "KFN CONTROLLED LATENT-OVERLAP "
        "ADAPTIVE CAPACITY-REUSE EXPERIMENT"
    )
    print("=" * 100)

    print("\nTask protocol:")

    for task_id, task_info in TASK_PROTOCOL.items():
        reference_task = task_info["reference_task"]

        if reference_task is None:
            overlap_text = "N/A"
        else:
            old_basis = set(
                TASK_PROTOCOL[reference_task][
                    "basis_indices"
                ]
            )

            new_basis = set(
                task_info["basis_indices"]
            )

            overlap = (
                len(old_basis & new_basis)
                / len(new_basis)
            )

            overlap_text = f"{overlap:.2f}"

        print(
            f"  T{task_id}: "
            f"basis={list(task_info['basis_indices'])} "
            f"| reference={reference_task} "
            f"| overlap={overlap_text} "
            f"| expected={task_info['expected']} "
            f"| role={task_info['role']}"
        )

    print("\nClear preregistered gate checks:")
    print("  T1: reuse")
    print("  T3: expand")
    print("  T4: reuse")


def print_summary(aggregated_results):
    print("\n" + "=" * 100)
    print("FINAL SUMMARY")
    print("=" * 100)

    header = (
        f"{'Configuration':<22} "
        f"{'AvgAcc':<14} "
        f"{'MinAcc':<14} "
        f"{'Forget':<14} "
        f"{'Plasticity':<14} "
        f"{'BWT':<14} "
        f"{'Exp':<10} "
        f"{'GateAcc':<14} "
        f"{'Params(M)':<14}"
    )

    print(header)
    print("-" * len(header))

    for configuration_name, values in (
        aggregated_results.items()
    ):
        print(
            f"{configuration_name:<22} "
            f"{values['avg_accuracy']['mean']:.2f}"
            f"±{values['avg_accuracy']['std']:.2f}{'':>2} "
            f"{values['min_accuracy']['mean']:.2f}"
            f"±{values['min_accuracy']['std']:.2f}{'':>2} "
            f"{values['forgetting']['mean']:.2f}"
            f"±{values['forgetting']['std']:.2f}{'':>2} "
            f"{values['plasticity']['mean']:.2f}"
            f"±{values['plasticity']['std']:.2f}{'':>2} "
            f"{values['bwt']['mean']:.2f}"
            f"±{values['bwt']['std']:.2f}{'':>2} "
            f"{values['num_expansions']['mean']:.2f}"
            f"±{values['num_expansions']['std']:.2f}{'':>2} "
            f"{values['clear_gate_accuracy']['mean']:.2f}"
            f"±{values['clear_gate_accuracy']['std']:.2f}{'':>2} "
            f"{values['final_total_params']['mean'] / 1e6:.3f}"
            f"±{values['final_total_params']['std'] / 1e6:.3f}"
        )


def main():
    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    os.makedirs(
        CONFIG.output_directory,
        exist_ok=True,
    )

    print(f"Device: {device}")

    print_protocol()

    configurations = [
        ("always_reuse", "always_reuse", 0.0),
        ("always_expand", "always_expand", 0.0),
    ]

    for threshold in CONFIG.threshold_values:
        configurations.append(
            (
                f"adaptive_tau_{threshold:.2f}",
                "adaptive",
                threshold,
            )
        )

    all_results = {}

    for configuration_name, policy, threshold in configurations:
        print("\n" + "=" * 80)
        print(
            f"CONFIGURATION: {configuration_name}"
        )
        print("=" * 80)

        seed_results = []

        for seed_index, seed in enumerate(
            CONFIG.seeds,
            start=1,
        ):
            print(
                f"\n[Seed {seed_index}/"
                f"{len(CONFIG.seeds)} = {seed}]"
            )

            result = run_single(
                policy=policy,
                threshold=threshold,
                seed=seed,
                config=CONFIG,
                device=device,
            )

            seed_results.append(result)

        all_results[configuration_name] = seed_results

    aggregated_results = aggregate_results(
        all_results
    )

    print_summary(aggregated_results)

    json_path = os.path.join(
        CONFIG.output_directory,
        "results.json",
    )

    output = {
        "config": asdict(CONFIG),
        "task_protocol": {
            str(task_id): {
                key: (
                    list(value)
                    if isinstance(value, tuple)
                    else value
                )
                for key, value in task_info.items()
            }
            for task_id, task_info in TASK_PROTOCOL.items()
        },
        "aggregated_results": aggregated_results,
        "per_seed_results": all_results,
    }

    with open(
        json_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            output,
            file,
            indent=2,
            default=str,
        )

    write_summary_csv(
        aggregated_results,
        CONFIG.output_directory,
    )

    write_transition_csv(
        all_results,
        CONFIG.output_directory,
    )

    print("\nSaved:")
    print(f"  {json_path}")
    print(
        "  "
        + os.path.join(
            CONFIG.output_directory,
            "summary.csv",
        )
    )
    print(
        "  "
        + os.path.join(
            CONFIG.output_directory,
            "transitions.csv",
        )
    )


if __name__ == "__main__":
    main()


Device: cuda
KFN CONTROLLED LATENT-OVERLAP ADAPTIVE CAPACITY-REUSE EXPERIMENT

Task protocol:
  T0: basis=[0, 1, 2, 3, 4, 5, 6, 7] | reference=None | overlap=N/A | expected=base | role=base
  T1: basis=[0, 1, 2, 3, 4, 5, 6, 7] | reference=0 | overlap=1.00 | expected=reuse | role=100% overlap with T0
  T2: basis=[4, 5, 6, 7, 8, 9, 10, 11] | reference=0 | overlap=0.50 | expected=intermediate | role=50% overlap with T0
  T3: basis=[16, 17, 18, 19, 20, 21, 22, 23] | reference=0 | overlap=0.00 | expected=expand | role=0% overlap with T0
  T4: basis=[16, 17, 18, 19, 20, 21, 22, 23] | reference=3 | overlap=1.00 | expected=reuse | role=100% overlap with T3

Clear preregistered gate checks:
  T1: reuse
  T3: expand
  T4: reuse

CONFIGURATION: always_reuse

[Seed 1/5 = 42]
  T0 [base                  ] overlap=N/A  latent_nu=1.0000 learned_nu=1.0000 gate=base   d_new=0   params=0.021M time=10.9s memory=17.0MB accs={T0:100.0}
  T1 [100% overlap with T0  ] overlap=1.00 latent_nu=0.0000 learned_nu=